# Feature Engineering Jurídico — Notebook Modular

Este notebook continua a partir da EDA forense e transforma a ABT exploratória em uma base mais forte para modelagem.

Ele usa os módulos pequenos do pacote:

```text
src/feature_engineering/
├── config.py
├── utils.py
├── value_features.py
├── date_features.py
├── event_features.py
├── entity_features.py
├── target_encoding.py
├── leakage.py
└── feature_selection.py
```

## Pré-requisito

Antes de rodar este notebook, execute o notebook da EDA ou o script `eda_forense_juridico.py`.

Arquivos esperados:

```text
data/processed/abt_eda_inicial.parquet
data/processed/eventos_tratados_eda.parquet
```

O arquivo `eventos_tratados_eda.parquet` é opcional para algumas etapas, mas recomendado.

## Objetivo

Criar features para um primeiro modelo/score jurídico:

1. Features financeiras.
2. Features de datas e safra.
3. Features derivadas da trajetória de eventos.
4. Features de entidades/listas, como advogados e partes.
5. Target encoding suavizado exploratório.
6. Remoção inicial de leakage.
7. Relatórios de força das features.
8. ABT final pronta para modelagem exploratória.

> Atenção: o target encoding aqui é exploratório. Para modelo oficial, deve ser out-of-fold ou point-in-time.

## 1. Imports básicos

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

## 2. Configurar path do projeto

In [ ]:
import sys
from pathlib import Path
import os

# Se este notebook estiver dentro da pasta notebooks/, a raiz do projeto é '..'
PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

print("Diretório atual:", os.getcwd())
print("PROJECT_ROOT:", PROJECT_ROOT)
print("Arquivos/pastas na raiz:")
print(list(PROJECT_ROOT.glob("*"))[:20])

## 3. Importar módulos de feature engineering

In [ ]:
from src.feature_engineering.config import (
    PROCESS_ID_COL,
    TARGET_COL,
    PROCESSED_DIR,
    INTERIM_DIR,
    REPORTS_DIR,
    MAIN_VALUE_COL,
    MAIN_DATE_COLS,
    LIST_ENTITY_COLS,
    GROUP_COLS_FOR_ENCODING,
    RECENT_WINDOWS,
)

from src.feature_engineering.value_features import (
    profile_value_columns,
    add_value_features,
)

from src.feature_engineering.date_features import (
    add_main_date_features,
)

from src.feature_engineering.event_features import (
    add_event_ratio_features,
    add_event_intensity_features,
    add_event_complexity_features,
    build_recent_event_features,
    build_last_n_event_sequences,
    build_transition_features,
)

from src.feature_engineering.entity_features import (
    add_list_entity_features,
)

from src.feature_engineering.target_encoding import (
    add_smooth_target_rate_features,
)

from src.feature_engineering.leakage import (
    identify_leakage_columns,
    remove_leakage_columns,
)

from src.feature_engineering.feature_selection import (
    split_feature_types,
    categorical_cardinality_report,
    numeric_feature_strength,
    categorical_feature_strength,
    build_feature_candidate_report,
)

print("Módulos importados com sucesso.")

### Caso o import falhe

Confirme se o ZIP `feature_engineering_juridico_modules.zip` foi extraído na raiz do projeto, mantendo a estrutura:

```text
src/feature_engineering/
```

## 4. Carregar ABT inicial da EDA

In [ ]:
abt_path = PROCESSED_DIR / "abt_eda_inicial.parquet"

print("Caminho esperado:", abt_path)
print("Existe?", abt_path.exists())

df_abt = pd.read_parquet(abt_path)

print(df_abt.shape)
df_abt.head()

### Checar target e chave

In [ ]:
print("PROCESS_ID_COL:", PROCESS_ID_COL)
print("TARGET_COL:", TARGET_COL)

print("Chave existe?", PROCESS_ID_COL in df_abt.columns)
print("Target existe?", TARGET_COL in df_abt.columns)

if TARGET_COL in df_abt.columns:
    display(df_abt[TARGET_COL].value_counts(dropna=False).to_frame("qtd"))

## 5. Profile das colunas de valor

Antes de criar features financeiras, vamos entender quais colunas de valor existem e quais têm maior cobertura.

Lembrete:

- `valor_valor` tende a ser uma feature candidata.
- `condenacao_valor`, `valor_arbitrado_valor`, `valor_do_acordo_valor` podem vazar o resultado.

In [ ]:
value_report = profile_value_columns(
    df_abt,
    output_path=REPORTS_DIR / "value_columns_profile.csv"
)

value_report

## 6. Criar features de valor

In [ ]:
print("Coluna principal de valor:", MAIN_VALUE_COL)
print("Existe na ABT?", MAIN_VALUE_COL in df_abt.columns)

df_abt = add_value_features(
    df_abt,
    value_col=MAIN_VALUE_COL
)

created_value_cols = [
    c for c in df_abt.columns
    if c.startswith(f"{MAIN_VALUE_COL}_")
]

created_value_cols

In [ ]:
df_abt[[MAIN_VALUE_COL] + created_value_cols].head()

**Interpretação**

As features criadas ajudam o modelo a capturar:

- processos sem valor informado;
- processos de valor zero;
- efeito não linear do valor;
- faixas de valor por quantil.

## 7. Criar features de datas e safra

In [ ]:
print("Datas principais configuradas:")
MAIN_DATE_COLS

In [ ]:
existing_date_cols = [c for c in MAIN_DATE_COLS if c in df_abt.columns]
missing_date_cols = [c for c in MAIN_DATE_COLS if c not in df_abt.columns]

print("Datas existentes:", existing_date_cols)
print("Datas ausentes:", missing_date_cols)

In [ ]:
df_abt, data_ref = add_main_date_features(
    df_abt,
    main_date_cols=MAIN_DATE_COLS,
    data_ref=None
)

print("Data de referência inferida:", data_ref)

In [ ]:
date_feature_cols = [
    c for c in df_abt.columns
    if any(suffix in c for suffix in ["_ano", "_mes", "_trimestre", "_ano_mes", "_is_null"])
    or c.startswith("idade_desde_")
]

date_feature_cols[:100], len(date_feature_cols)

In [ ]:
df_abt[date_feature_cols].head()

**Interpretação**

Essas features capturam:

- safra do processo;
- idade do processo;
- cobertura/ausência de datas importantes;
- possíveis mudanças temporais de comportamento.

## 8. Features de proporção de eventos

In [ ]:
event_count_cols = [
    c for c in df_abt.columns
    if c.startswith("qtd_evento_cat_")
]

print("Quantidade de colunas de contagem de evento:", len(event_count_cols))
event_count_cols[:50]

In [ ]:
df_abt = add_event_ratio_features(df_abt)

event_ratio_cols = [
    c for c in df_abt.columns
    if c.startswith("perc_evento_cat_")
]

print("Features de proporção criadas:", len(event_ratio_cols))
event_ratio_cols[:50]

In [ ]:
df_abt[event_ratio_cols].head()

**Interpretação**

As proporções complementam as contagens.

Exemplo:

- 3 recursos em 5 eventos pode ser mais relevante do que 3 recursos em 300 eventos.

## 9. Features de intensidade de eventos

In [ ]:
df_abt = add_event_intensity_features(df_abt)

intensity_cols = [
    "qtd_eventos_log1p",
    "flag_sem_eventos",
    "qtd_eventos_faixa_q",
]

[c for c in intensity_cols if c in df_abt.columns]

In [ ]:
existing_intensity_cols = [c for c in intensity_cols if c in df_abt.columns]
df_abt[["qtd_eventos"] + existing_intensity_cols].head()

**Interpretação**

- `qtd_eventos_log1p`: reduz o efeito de cauda longa.
- `flag_sem_eventos`: captura processos sem movimentação disponível.
- `qtd_eventos_faixa_q`: cria faixas de intensidade.

## 10. Score simples de complexidade processual

In [ ]:
df_abt = add_event_complexity_features(df_abt)

complexity_cols = [
    c for c in df_abt.columns
    if "complexidade" in c or c in [
        "qtd_categorias_evento_distintas_log1p",
        "dias_entre_primeiro_ultimo_evento_log1p"
    ]
]

complexity_cols

In [ ]:
df_abt[complexity_cols].head()

In [ ]:
if "score_complexidade_processual_simples" in df_abt.columns:
    display(df_abt["score_complexidade_processual_simples"].describe())

**Interpretação**

Esse score não é score de risco. Ele mede complexidade operacional/processual com base em:

- volume de eventos;
- diversidade de categorias;
- duração da trajetória.

## 11. Features de entidades e listas

In [ ]:
print("Colunas de lista configuradas:")
LIST_ENTITY_COLS

existing_list_cols = [c for c in LIST_ENTITY_COLS if c in df_abt.columns]
missing_list_cols = [c for c in LIST_ENTITY_COLS if c not in df_abt.columns]

print("Existentes:", existing_list_cols)
print("Ausentes:", missing_list_cols)

In [ ]:
df_abt = add_list_entity_features(
    df_abt,
    list_cols=LIST_ENTITY_COLS
)

entity_feature_cols = []
for col in existing_list_cols:
    entity_feature_cols.extend([
        f"{col}_parsed",
        f"qtd_{col}",
        f"primeiro_{col}",
    ])

entity_feature_cols = [c for c in entity_feature_cols if c in df_abt.columns]
entity_feature_cols

In [ ]:
df_abt[entity_feature_cols].head()

**Interpretação**

Essas features ajudam a capturar:

- quantidade de advogados do requerente;
- quantidade de advogados do requerido;
- quantidade de partes;
- primeira entidade normalizada para agregações futuras.

Cuidado: nomes de advogados e partes têm alta cardinalidade.

## 12. Opcional — Carregar eventos tratados para features adicionais

Esta seção usa `eventos_tratados_eda.parquet`, gerado pela EDA.

Se o arquivo não existir, pule as próximas etapas opcionais.

In [ ]:
eventos_path = PROCESSED_DIR / "eventos_tratados_eda.parquet"

print("Caminho:", eventos_path)
print("Existe?", eventos_path.exists())

if eventos_path.exists():
    df_eventos = pd.read_parquet(eventos_path)
    print(df_eventos.shape)
    display(df_eventos.head())
else:
    df_eventos = None
    print("Arquivo de eventos tratados não encontrado. Pule as etapas opcionais com eventos.")

## 13. Opcional — Features de eventos recentes

In [ ]:
if df_eventos is not None:
    recent_event_features = build_recent_event_features(
        df_eventos,
        process_id_col=PROCESS_ID_COL,
        date_col="data",
        category_col="tipo_categoria",
        windows=RECENT_WINDOWS,
        data_ref=data_ref,
    )

    print(recent_event_features.shape)
    display(recent_event_features.head())

    recent_event_features.to_parquet(
        PROCESSED_DIR / "features_eventos_recentes.parquet",
        index=False
    )
else:
    print("df_eventos não carregado. Etapa pulada.")

In [ ]:
if df_eventos is not None:
    df_abt = df_abt.merge(
        recent_event_features,
        on=PROCESS_ID_COL,
        how="left"
    )

    recent_cols = [c for c in recent_event_features.columns if c != PROCESS_ID_COL]
    df_abt[recent_cols] = df_abt[recent_cols].fillna(0)

    print(df_abt.shape)
else:
    print("Merge de eventos recentes pulado.")

**Interpretação**

Essas features capturam atividade recente:

- eventos nos últimos 30/60/90/180/365 dias;
- categorias específicas recentes, como recurso, execução, acordo etc.

## 14. Opcional — Sequência dos últimos eventos

In [ ]:
if df_eventos is not None:
    sequence_features = build_last_n_event_sequences(
        df_eventos,
        process_id_col=PROCESS_ID_COL,
        date_col="data",
        category_col="tipo_categoria",
        n_values=[3, 5, 10],
    )

    print(sequence_features.shape)
    display(sequence_features.head())

    sequence_features.to_parquet(
        PROCESSED_DIR / "features_eventos_sequencias.parquet",
        index=False
    )
else:
    print("df_eventos não carregado. Etapa pulada.")

In [ ]:
if df_eventos is not None:
    df_abt = df_abt.merge(
        sequence_features,
        on=PROCESS_ID_COL,
        how="left"
    )

    print(df_abt.shape)
else:
    print("Merge de sequências pulado.")

**Interpretação**

As sequências ajudam a entender o estado recente do processo.

Exemplo:

```text
sentenca > intimacao > recurso > juntada_peticao > conclusos
```

Inicialmente são mais úteis para análise e explicabilidade. Futuramente podem virar TF-IDF ou embeddings.

## 15. Opcional — Transições entre eventos

In [ ]:
if df_eventos is not None:
    transition_features, transition_freq = build_transition_features(
        df_eventos,
        process_id_col=PROCESS_ID_COL,
        date_col="data",
        category_col="tipo_categoria",
        top_n=50,
    )

    print("Features de transição:", transition_features.shape)
    display(transition_features.head())

    print("Frequência das transições:")
    display(transition_freq.head(50))

    transition_features.to_parquet(
        PROCESSED_DIR / "features_eventos_transicoes.parquet",
        index=False
    )

    transition_freq.to_csv(
        REPORTS_DIR / "transition_frequency_report.csv",
        index=False
    )
else:
    print("df_eventos não carregado. Etapa pulada.")

In [ ]:
if df_eventos is not None:
    df_abt = df_abt.merge(
        transition_features,
        on=PROCESS_ID_COL,
        how="left"
    )

    transition_cols = [c for c in transition_features.columns if c != PROCESS_ID_COL]
    df_abt[transition_cols] = df_abt[transition_cols].fillna(0)

    print(df_abt.shape)
else:
    print("Merge de transições pulado.")

**Interpretação**

Transições capturam trajetória, não apenas presença de eventos.

Exemplos:

- `sentenca_para_recurso`;
- `audiencia_para_acordo`;
- `execucao_para_pagamento`;
- `citacao_para_contestacao`.

Cuidado com leakage se a previsão for feita no início do processo.

## 16. Salvar ABT completa de feature engineering

In [ ]:
df_abt.to_parquet(
    PROCESSED_DIR / "abt_feature_engineering_juridico.parquet",
    index=False
)

print("ABT completa salva em:", PROCESSED_DIR / "abt_feature_engineering_juridico.parquet")
print(df_abt.shape)

## 17. Target encoding suavizado exploratório

Esta etapa cria taxas históricas suavizadas de perda por grupos jurídicos, como assunto, vara, UF, órgão julgador, escritório e última categoria de evento.

> Atenção: isso é exploratório. Para modelo oficial, o cálculo deve ser out-of-fold ou point-in-time.

In [ ]:
if TARGET_COL not in df_abt.columns:
    raise KeyError(f"Target {TARGET_COL} não encontrado na ABT.")

print(df_abt[TARGET_COL].value_counts(dropna=False))

In [ ]:
encoding_cols = [c for c in GROUP_COLS_FOR_ENCODING if c in df_abt.columns]

print("Colunas disponíveis para encoding:", len(encoding_cols))
encoding_cols

In [ ]:
df_abt_encoded, encoding_reports = add_smooth_target_rate_features(
    df_abt,
    group_cols=encoding_cols,
    target_col=TARGET_COL,
    k=30,
    min_count=1,
    reports_dir=REPORTS_DIR,
)

print(df_abt_encoded.shape)

In [ ]:
encoding_feature_cols = [
    c for c in df_abt_encoded.columns
    if c.startswith("taxa_perda_suavizada_") or c.startswith("qtd_hist_")
]

encoding_feature_cols[:100], len(encoding_feature_cols)

In [ ]:
df_abt_encoded[encoding_feature_cols].head()

**Interpretação**

Essas features capturam histórico suavizado por entidade/grupo.

Exemplo:

- taxa histórica de perda por assunto;
- taxa histórica de perda por vara;
- taxa histórica de perda por órgão julgador;
- taxa histórica de perda por escritório;
- taxa histórica de perda por última categoria de evento.

O parâmetro `k=30` puxa grupos pequenos para a média global, reduzindo ruído.

## 18. Identificar colunas com possível leakage

In [ ]:
leakage_cols = identify_leakage_columns(df_abt_encoded.columns)

print("Quantidade de colunas candidatas a leakage:", len(leakage_cols))
leakage_cols[:200]

In [ ]:
pd.DataFrame({"coluna": leakage_cols}).to_csv(
    REPORTS_DIR / "leakage_columns_identified_feature_engineering.csv",
    index=False
)

**Atenção**

A lista é conservadora. Revise antes de excluir tudo cegamente.

Exemplos normalmente perigosos:

- resultado;
- sentença;
- acórdão;
- condenação;
- trânsito em julgado;
- valor de acordo;
- valor arbitrado;
- predições externas `_prob`;
- campos `provavel_*`;
- links e campos deprecated.

## 19. Remover leakage para criar ABT modelável

In [ ]:
df_model_abt, removed_leakage_cols = remove_leakage_columns(
    df_abt_encoded,
    target_col=TARGET_COL,
    keep_cols=[PROCESS_ID_COL, "target_source"],
)

print("ABT com todas as features:", df_abt_encoded.shape)
print("ABT modelável sem leakage:", df_model_abt.shape)
print("Colunas removidas:", len(removed_leakage_cols))

In [ ]:
pd.DataFrame({"coluna": removed_leakage_cols}).to_csv(
    REPORTS_DIR / "leakage_columns_to_remove_features_step.csv",
    index=False
)

removed_leakage_cols[:200]

**Interpretação**

Esta ABT sem leakage é a candidata para um primeiro modelo exploratório.

Mesmo assim, ainda é necessário revisar se as features de eventos representam score atual ou previsão em uma data passada.

## 20. Separar tipos de features

In [ ]:
feature_cols, numeric_features, categorical_features = split_feature_types(
    df_model_abt,
    id_cols=[PROCESS_ID_COL],
    target_cols=[TARGET_COL, "target_source"],
)

print("Total de features:", len(feature_cols))
print("Features numéricas:", len(numeric_features))
print("Features categóricas:", len(categorical_features))

In [ ]:
numeric_features[:100]

In [ ]:
categorical_features[:100]

## 21. Cardinalidade das categóricas

In [ ]:
cat_cardinality = categorical_cardinality_report(
    df_model_abt,
    categorical_features
)

cat_cardinality.head(100)

In [ ]:
cat_cardinality.to_csv(
    REPORTS_DIR / "categorical_feature_cardinality_model_abt.csv",
    index=False
)

**Interpretação**

Categóricas com cardinalidade muito alta não devem entrar diretamente em one-hot no primeiro modelo.

Alternativas:

- target encoding suavizado;
- frequência histórica;
- top N + outros;
- embeddings ou hashing no futuro.

## 22. Força das features numéricas

In [ ]:
num_strength = numeric_feature_strength(
    df_model_abt,
    numeric_features,
    target_col=TARGET_COL
)

num_strength.head(100)

In [ ]:
num_strength.to_csv(
    REPORTS_DIR / "numeric_feature_strength.csv",
    index=False
)

**Interpretação**

Analise:

- cobertura da feature;
- diferença de média entre perda e não perda;
- correlação absoluta com target;
- plausibilidade jurídica.

## 23. Força das features categóricas

In [ ]:
safe_categorical_features = (
    cat_cardinality
    .query("qtd_distintos <= 100")
    ["coluna"]
    .tolist()
)

print("Categóricas seguras para análise inicial:", len(safe_categorical_features))
safe_categorical_features[:100]

In [ ]:
cat_strength = categorical_feature_strength(
    df_model_abt,
    safe_categorical_features,
    target_col=TARGET_COL,
    min_count=30
)

cat_strength.head(100)

In [ ]:
cat_strength.to_csv(
    REPORTS_DIR / "categorical_feature_strength.csv",
    index=False
)

**Interpretação**

Features categóricas fortes são aquelas que mostram grande variação de taxa de perda entre categorias.

Exemplo:

- assunto A com taxa de perda muito acima da média;
- vara B com taxa muito abaixo da média;
- fase processual específica com comportamento distinto.

## 24. Gerar relatório final de features candidatas

In [ ]:
feature_reports = build_feature_candidate_report(
    df_model_abt,
    id_cols=[PROCESS_ID_COL],
    target_cols=[TARGET_COL, "target_source"],
    target_col=TARGET_COL,
    max_cat_cardinality=100,
    min_count=30,
    reports_dir=REPORTS_DIR,
)

candidate_features = feature_reports["candidate_features"]

print("Quantidade de features candidatas:", len(candidate_features))
candidate_features[:100]

In [ ]:
feature_reports["feature_ranking"].head(100)

## 25. Criar ABT final para modelagem exploratória

Esta base contém:

- ID do processo;
- target;
- target_source, se existir;
- features candidatas selecionadas pelos relatórios.

In [ ]:
final_cols = [PROCESS_ID_COL, TARGET_COL, "target_source"] + candidate_features
final_cols = [c for c in final_cols if c in df_model_abt.columns]

df_model_ready = df_model_abt[final_cols].copy()

print(df_model_ready.shape)
df_model_ready.head()

In [ ]:
df_model_ready.to_parquet(
    PROCESSED_DIR / "abt_model_ready_juridico.parquet",
    index=False
)

df_model_abt.to_parquet(
    PROCESSED_DIR / "abt_model_abt_sem_leakage.parquet",
    index=False
)

df_abt_encoded.to_parquet(
    PROCESSED_DIR / "abt_feature_engineering_juridico_encoded.parquet",
    index=False
)

print("Arquivos salvos:")
print(PROCESSED_DIR / "abt_model_ready_juridico.parquet")
print(PROCESSED_DIR / "abt_model_abt_sem_leakage.parquet")
print(PROCESSED_DIR / "abt_feature_engineering_juridico_encoded.parquet")

## 26. Validar saídas esperadas

In [ ]:
expected_outputs = [
    PROCESSED_DIR / "abt_feature_engineering_juridico.parquet",
    PROCESSED_DIR / "abt_feature_engineering_juridico_encoded.parquet",
    PROCESSED_DIR / "abt_model_abt_sem_leakage.parquet",
    PROCESSED_DIR / "abt_model_ready_juridico.parquet",
    REPORTS_DIR / "value_columns_profile.csv",
    REPORTS_DIR / "leakage_columns_to_remove_features_step.csv",
    REPORTS_DIR / "categorical_feature_cardinality_model_abt.csv",
    REPORTS_DIR / "numeric_feature_strength.csv",
    REPORTS_DIR / "categorical_feature_strength.csv",
    REPORTS_DIR / "candidate_features_for_model.csv",
]

for path in expected_outputs:
    print(path, "OK" if path.exists() else "NÃO ENCONTRADO")

## 27. Como interpretar os outputs

Leia os arquivos nesta ordem:

1. `value_columns_profile.csv`
2. `leakage_columns_identified_feature_engineering.csv`
3. `leakage_columns_to_remove_features_step.csv`
4. `categorical_feature_cardinality_model_abt.csv`
5. `numeric_feature_strength.csv`
6. `categorical_feature_strength.csv`
7. `candidate_features_for_model.csv`

A base principal para o próximo passo é:

```text
data/processed/abt_model_ready_juridico.parquet
```

## 28. Narrativa executiva desta etapa

Nesta etapa, transformamos a ABT exploratória em uma base mais preparada para modelagem.

Criamos features financeiras, temporais, de trajetória processual, de intensidade de eventos, de complexidade processual, de entidades jurídicas e de histórico suavizado por grupos como assunto, vara, órgão julgador e fase.

Também fizemos uma remoção inicial de colunas com risco de vazamento, incluindo resultados, sentenças, acórdãos, condenações, valores posteriores ao desfecho e predições externas.

O resultado é uma ABT modelável e um conjunto de relatórios que ajudam a priorizar as melhores features para um primeiro modelo de risco jurídico.